# Retrieving without looking at every book

Unit 8's two-tower is the book's best recommender so far: hit@10 ~0.340 on `val`, by learning a taste
vector for every reader and every book and ranking their dot product. Unit 9 gave cold books a vector
too. But every dense path we have built -- U7 semantic, U8 two-tower, U9 feature tower -- answers one
reader the same expensive way: **score all 2,000 books, then sort**. One `reader . item` dot product
per book, per reader.

That is fine at 2,000 books. It is hopeless at a million: a full scan per query does not scale. So
this unit is not a new model -- it is about the **retrieval infrastructure**. Can we retrieve the
*same* books without looking at all of them? We build the **exact brute-force baseline**, replace it
with an **approximate** index (FAISS **HNSW**), and measure the one trade that matters -- **recall vs
speed**. Then we try a **hybrid** path that fuses Unit 3's lexical BM25 with the dense two-tower, and
read -- honestly -- whether fusing helps.

## Load the substrate and fit the dense source

The Units 1-9 substrate is unchanged: `generated_dir()` locates the seeded slice, and the catalog, the
gzip'd interaction log and the keyword corpus load as before. `cold_partitions.json` lists the cold
readers we always exclude from scoring, and `test` stays sealed -- we only ever score on `val`.

The dense source this unit indexes is Unit 8's two-tower. We fit it **once** here (seed 0, the shipped
configuration) and reuse the same fitted model everywhere below -- its **item matrix** is what the
index is built over, and its **reader vectors** are the queries.

In [ ]:
import json
from collections import defaultdict

import numpy as np
import pandas as pd
from bookrec import (
    AnnRetrievalPath,
    HnswIndex,
    HybridRetrievalPath,
    LexicalRetrievalPath,
    TwoTowerRetrievalPath,
    generated_dir,
    load_catalog,
    load_keywords,
    run_blended_scoreboard,
    run_validation_scoreboard,
)

data_dir = generated_dir()
catalog = load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
rows = pd.read_csv(interactions_path).to_dict("records")
keywords = load_keywords(data_dir / "keywords.csv.gz")
cold_readers = set(json.loads((data_dir / "cold_partitions.json").read_text())["cold_readers"])

# Each reader's train history (the books they have already read) -- positive train rows only.
seen_by = defaultdict(set)
for row in rows:
    if int(row["label"]) == 1 and row["split"] == "train":
        seen_by[int(row["reader_id"])].add(int(row["item_id"]))

# Fit Unit 8's two-tower ONCE (seed 0, the shipped dense source) and reuse it everywhere below.
two_tower = TwoTowerRetrievalPath(seed=0).fit(rows, catalog=catalog_ids)
art = two_tower.artifact()
item_matrix = art["item_embeddings"]      # (2000, 32) float32 -- one row per book
reader_matrix = art["reader_embeddings"]  # (n_readers, 32) float32 -- one row per fitted reader
item_ids = [int(i) for i in art["item_ids"]]
reader_ids = [int(r) for r in art["reader_ids"]]
reader_of = {r: i for i, r in enumerate(reader_ids)}
print(f"catalog {len(catalog_ids)} books; {len(reader_ids)} fitted readers; item vectors {item_matrix.shape}")

catalog 2000 books; 540 fitted readers; item vectors (2000, 32)


## 1. The exact brute-force baseline

Start with what every dense path already does inside `retrieve()`. To rank books for a reader it
computes `reader . item` for **every** book -- 2,000 dot products -- and sorts the scores. That is the
exact top-k: nothing approximated, nothing missed. It is also **linear in the catalog** -- double the
books and you double the work, for every single query.

In [ ]:
# The exact brute-force baseline: score EVERY book for a reader, then sort. This is exactly what
# every dense path (U7 semantic, U8 two-tower, U9 feature tower) does inside retrieve().
reader_id = reader_ids[0]
row = reader_of[reader_id]
scores = reader_matrix[row] @ item_matrix.T   # one reader . item dot product per book -> 2000 scores
order = np.argsort(-scores)[:10]              # sort all 2000 and keep the top 10 (O(catalog))
exact_ids = [item_ids[j] for j in order]
print(f"reader {reader_id}: scored all {scores.shape[0]} books with one dot product each")
print(f"exact top-10 book ids: {exact_ids}")

# The shipped two-tower retrieve() is the same scan, with the reader's already-read books removed.
recs = two_tower.retrieve(reader_id, {"seen": seen_by[reader_id]}, 10)
print(f"two-tower top-10 (seen removed): {[c.item_id for c in recs]}")
print("Scoring all 2000 is fine here. At a million books, one dot product per book per reader is hopeless.")

reader 0: scored all 2000 books with one dot product each
exact top-10 book ids: [296, 518, 1922, 651, 1324, 253, 834, 527, 157, 1429]
two-tower top-10 (seen removed): [1922, 1324, 834, 157, 1429, 1605, 71, 524, 622, 1786]
Scoring all 2000 is fine here. At a million books, one dot product per book per reader is hopeless.


## 2. HNSW: a navigable graph instead of a full scan

The fix is to stop scoring every book. **HNSW** (Hierarchical Navigable Small World) stores the item
vectors as a layered graph whose edges link near neighbours. A search enters at one node and **walks
greedily toward the query**, visiting a few dozen nodes instead of all 2,000 -- so query time grows
far slower than the catalog does.

We build it with FAISS. The one detail that matters: FAISS `IndexHNSWFlat` defaults to **L2** distance,
but the two-tower ranks by **inner product**, so we build it `METRIC_INNER_PRODUCT` -- otherwise the
graph ranks by the wrong geometry and recall-vs-exact is meaningless. The shipped `HnswIndex` wraps
exactly that call:

```
import faiss
index = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)  # M = 32 graph degree
index.hnsw.efConstruction = 200
index.add(item_matrix)                                            # single-threaded, deterministic
```

`M` is the graph degree and `efConstruction` how hard the build works; both are pinned. `efSearch` --
set at query time -- is the **recall/speed knob** we tune next. The build is single-threaded
(`faiss.omp_set_num_threads(1)`, saved and restored) so two builds are reproducible.

In [ ]:
# Build the HNSW graph over the SAME two-tower item vectors (inner-product metric, pinned params).
index = HnswIndex(item_matrix, item_ids, m=32, ef_construction=200)
print(f"indexed {index.size} book vectors in a navigable graph (dim {item_matrix.shape[1]})")

# Query the graph for ONE reader -- it visits a few neighbours instead of all 2000 items.
query = reader_matrix[row:row + 1]                      # shape (1, 32): the reader's vector
labels, approx_scores = index.search(query, 10, efSearch=64)
ann_ids = [item_ids[j] for j in labels[0] if j >= 0]
overlap = len(set(ann_ids) & set(exact_ids))
print(f"ANN top-10 book ids: {ann_ids}")
print(f"overlap with the exact top-10: {overlap} of 10")

indexed 2000 book vectors in a navigable graph (dim 32)
ANN top-10 book ids: [296, 518, 1922, 651, 1324, 253, 834, 527, 157, 1429]
overlap with the exact top-10: 10 of 10


## 3. ANN recall vs exact, and the efSearch knob

An approximate search can miss a true neighbour, so we measure it honestly against the exact baseline:
**recall@10 vs exact** is the fraction of each reader's exact top-10 that the ANN search also returns,
averaged over every fitted reader. `efSearch` is the knob -- raise it and the search explores more of
the graph, recovering more of the exact top-10 at the cost of speed. Sweep it from cheap to generous
and watch recall climb toward 1.0.

In [ ]:
def exact_top_rows(readers, items, k):
    # Exact top-k item ROW indices per reader by inner product -- the brute-force reference.
    full_scores = readers @ items.T
    return np.argsort(-full_scores, axis=1)[:, :k]


def ann_recall(idx, readers, exact_rows, ef):
    # Mean per-reader overlap@k between the ANN neighbours and the exact top-k (both in row space).
    found, _ = idx.search(readers, exact_rows.shape[1], efSearch=ef)
    k = exact_rows.shape[1]
    hits = []
    for i in range(readers.shape[0]):
        ann_rows = {int(x) for x in found[i] if x >= 0}
        exact_set = {int(x) for x in exact_rows[i]}
        hits.append(len(ann_rows & exact_set) / k)
    return float(np.mean(hits))


exact_rows = exact_top_rows(reader_matrix, item_matrix, 10)
print(f"{'efSearch':>9} {'recall@10 vs exact':>20}")
for ef in (8, 16, 32, 64, 128):
    print(f"{ef:>9} {ann_recall(index, reader_matrix, exact_rows, ef):>20.4f}")
print()
print("efSearch is the recall/speed knob: higher explores more of the graph (higher recall, slower).")
print("At efSearch=64 recall is ~0.999 -- the approximate search recovers almost the entire exact top-10.")

 efSearch   recall@10 vs exact
        8               0.9813
       16               0.9963
       32               0.9983
       64               0.9991
      128               0.9996

efSearch is the recall/speed knob: higher explores more of the graph (higher recall, slower).
At efSearch=64 recall is ~0.999 -- the approximate search recovers almost the entire exact top-10.


### Reproducible, not random

An approximate index is not a *random* one. Two independent single-thread builds give the **same**
neighbours -- identical ids and `allclose` scores. That reproducibility, together with recall-vs-exact,
is the gate we hold the index to; we never demand the ANN neighbours match the brute force *exactly*,
because by construction they need not.

In [ ]:
# Design 011 section 184: two independent single-thread builds must agree (ids + allclose scores).
build_a = HnswIndex(item_matrix, item_ids, m=32, ef_construction=200)
build_b = HnswIndex(item_matrix, item_ids, m=32, ef_construction=200)
labels_a, scores_a = build_a.search(reader_matrix, 10, efSearch=64)
labels_b, scores_b = build_b.search(reader_matrix, 10, efSearch=64)
print(f"identical neighbour ids: {np.array_equal(labels_a, labels_b)}")
print(f"allclose scores:         {np.allclose(scores_a, scores_b)}")

identical neighbour ids: True
allclose scores:         True


## 4. ANN preserves accuracy -- it is a speed technique

Recall ~0.999 is high, but does the handful of missed neighbours change the recommendations a reader
actually sees? Wrap the two-tower in the shipped `AnnRetrievalPath` -- it builds the HNSW index over
the dense source's item matrix in `fit()` and answers `retrieve()` by an approximate search, over-
fetching past the reader's `seen` books so parity with the exact scan holds -- and score it on the
same frozen `val` scoreboard. If ANN recovers the same top-k, its hit@10 should match the exact
two-tower's.

In [ ]:
ann = AnnRetrievalPath(two_tower, efSearch=64, m=32, ef_construction=200).fit(rows, catalog=catalog_ids)

kwargs = dict(catalog_ids=catalog_ids, k=10, cold_readers=cold_readers)
tt_hit = run_validation_scoreboard(two_tower, interactions_path, **kwargs).hit_rate_at_k
ann_hit = run_validation_scoreboard(ann, interactions_path, **kwargs).hit_rate_at_k
print(f"exact two-tower hit@10: {tt_hit:.4f}")
print(f"ANN hit@10:             {ann_hit:.4f}  (delta {ann_hit - tt_hit:+.4f})")
print()
print("ANN is a SPEED technique, not an accuracy one: its hit@10 matches the exact two-tower.")
print("It recovers the same recommendations -- it does not make them better.")

exact two-tower hit@10: 0.3400
ANN hit@10:             0.3400  (delta +0.0000)

ANN is a SPEED technique, not an accuracy one: its hit@10 matches the exact two-tower.
It recovers the same recommendations -- it does not make them better.


## 5. Latency, honestly: the win is asymptotic

Here is the part many tutorials get wrong. At **2,000 items the exact scan is already microseconds per
reader** -- a 2,000 x 32 matrix multiply is tiny -- so HNSW is *not* faster here; the graph's per-query
bookkeeping is not yet repaid. The win is **asymptotic**: it shows when the catalog is large. The exact
scan must score **every** item, so its work grows *with the catalog*; the graph search visits only a
bounded neighbourhood, so its per-query work stays nearly flat. Grow the catalog to 20,000 (jittered
copies of the item vectors) and the gap opens -- a few times faster here, widening toward an order of
magnitude at the hundreds-of-thousands scale real catalogs reach (too slow to build inside CI). Watch
recall, too: at fixed `efSearch` it slips as the catalog grows, so the knob must scale with the catalog.

(Wall-clock timings vary run to run, so this cell reports the deterministic facts -- the work each
method does, the under-a-second cohort query, and the measured 20k recall -- not raw milliseconds.)

In [ ]:
from time import perf_counter


def ann_cohort_seconds(idx, readers, k, ef, repeats=5):
    # Best of a few batched ANN queries over the whole reader cohort (seconds).
    best = None
    for _ in range(repeats):
        start = perf_counter()
        idx.search(readers, k, efSearch=ef)
        elapsed = perf_counter() - start
        best = elapsed if best is None else min(best, elapsed)
    return best


# Grow the catalog to 20,000 with seeded, jittered copies of the item vectors (so it is reproducible).
rng = np.random.default_rng(0)
copies = [item_matrix]
for _ in range(9):
    copies.append((item_matrix + rng.normal(0.0, 0.01, size=item_matrix.shape)).astype(np.float32))
big_items = np.concatenate(copies, axis=0)   # (20000, 32)
big_index = HnswIndex(big_items, range(big_items.shape[0]), m=32, ef_construction=200)

exact_rows_20k = exact_top_rows(reader_matrix, big_items, 10)
recall_20k = ann_recall(big_index, reader_matrix, exact_rows_20k, 64)
cohort_under_1s = ann_cohort_seconds(big_index, reader_matrix, 10, 64) < 1.0

print(f"exact scan scores EVERY item per reader: {item_matrix.shape[0]} at 2k, "
      f"{big_items.shape[0]} at 20k (10x the catalog -> 10x the dot products)")
print("ANN visits only a bounded neighbourhood of the graph, so its per-query work stays nearly flat.")
print(f"ANN query over all {reader_matrix.shape[0]} readers at 20k finished under 1 s: {cohort_under_1s}")
print(f"20k ANN recall@10 vs exact: {recall_20k:.4f}   "
      f"(vs ~0.999 at 2k -- recall falls as the catalog grows at fixed efSearch)")

exact scan scores EVERY item per reader: 2000 at 2k, 20000 at 20k (10x the catalog -> 10x the dot products)
ANN visits only a bounded neighbourhood of the graph, so its per-query work stays nearly flat.
ANN query over all 540 readers at 20k finished under 1 s: True
20k ANN recall@10 vs exact: 0.9922   (vs ~0.999 at 2k -- recall falls as the catalog grows at fixed efSearch)


## 6. Hybrid: fuse the sparse and dense paths

ANN made the dense path *fast*. A different question: can we make it *better* by fusing it with a
completely different signal? Unit 3's BM25 retrieves by **lexical** keyword overlap; the two-tower
retrieves by **learned taste**. They surface largely different books. `HybridRetrievalPath` fuses them:
ask each path for its top `pool=50` candidates, **min-max-normalise** each path's scores so they are
comparable, then take a weighted sum (`weight=0.7` on the dense path -- dense-heavy, because the
two-tower is the stronger path). Read **both** hit@10 and catalog coverage, and compare against naive
fusions (equal weights, and reciprocal-rank fusion).

In [ ]:
def board(path):
    # hit@10 + catalog coverage via the single-path blended scoreboard (reproduces the path's hit@10).
    return run_blended_scoreboard(
        {path.name: path}, interactions_path,
        catalog_ids=catalog_ids, k=10, pool=10, cold_readers=cold_readers,
    )


lexical = LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
hybrid = HybridRetrievalPath(lexical, two_tower, weight=0.7, pool=50, method="weighted").fit(rows)
equal = HybridRetrievalPath(lexical, two_tower, weight=0.5, pool=50, method="weighted").fit(rows)
rrf = HybridRetrievalPath(lexical, two_tower, weight=0.5, pool=50, method="rrf").fit(rows)

print(f"{'path':>28} {'hit@10':>8} {'coverage':>10}")
for label, path in (
    ("BM25 lexical (U3)", lexical),
    ("two-tower (U8)", two_tower),
    ("hybrid w0.7 pool50 (PINNED)", hybrid),
    ("hybrid equal weights", equal),
    ("hybrid RRF", rrf),
):
    b = board(path)
    print(f"{label:>28} {b.hit_rate_at_k:>8.3f} {b.catalog_coverage:>10.3f}")
print(f"{'U6 4-way blend (recorded)':>28} {0.306:>8.3f} {0.333:>10.3f}")

                        path   hit@10   coverage
           BM25 lexical (U3)    0.158      0.278
              two-tower (U8)    0.340      0.177
 hybrid w0.7 pool50 (PINNED)    0.362      0.192
        hybrid equal weights    0.308      0.242
                  hybrid RRF    0.328      0.215
   U6 4-way blend (recorded)    0.306      0.333


Read the table honestly -- that is the whole point of this section.

- The pinned **dense-heavy hybrid** (`weight=0.7`) edges the two-tower on **both** axes: hit@10 ~0.362
  vs 0.340, coverage ~0.192 vs 0.177. BM25 rescues a few lexically on-the-nose books the dense tower
  ranked just below the cut.
- But the lift is **small and tuning-dependent**. **Equal weights hurt** (hit drops to ~0.30) and
  **reciprocal-rank fusion loses** to the two-tower alone. Fusion is not a free win -- a careless
  fusion is a *loss*.
- And coverage, while up a little, stays **well below** BM25 alone (~0.278) and the Unit 6 blend
  (~0.333). The hybrid is a *modest complement*, not a new best.

As always: one read, on `val`, at this pinned config, with `test` sealed.

## Where this goes

- Every dense path retrieves by an **exact brute-force scan** -- one `reader . item` dot product per
  book, then a sort. It is exact but **linear in the catalog**: fine at 2,000 books, hopeless at a
  million.
- **HNSW** replaces the scan with a **navigable graph** that visits a few neighbours instead of all
  items. Built `METRIC_INNER_PRODUCT` (to match the dot product) and single-threaded (to be
  reproducible), it recovers the exact top-10 at **recall ~0.999** at `efSearch=64` -- and `efSearch`
  is the recall/speed knob.
- **ANN preserves accuracy**: its hit@10 equals the exact two-tower (0.340). It is a **speed**
  technique, and the speed win is **asymptotic** -- no edge at 2k, a few times faster at 20k and an
  order of magnitude at larger scale -- with recall falling as the catalog grows at fixed `efSearch`.
- **Hybrid** fusion of BM25 + the two-tower gives only a **small, tuning-dependent** lift (~0.362 /
  coverage ~0.192): equal weights hurt and RRF loses, and coverage stays below lexical and the U6
  blend. A modest complement, not a new best.

### The bridge ahead

Retrieval is only the first stage. **Unit 11** adds a learned **reranker** on top of the shortlist the
index returns -- retrieve a pool fast, then rank it well. **Unit 12** brings **sequence models**, which
read the *order* of a reader's history rather than a bag of it. The retrieve-then-rank split this unit
makes concrete is the backbone of every stage to come.